In [ ]:
from pathlib import Path
from tqdm.notebook import tqdm

from combra import data, angles

# Grid data generation

Unified pipeline: orig (ethalon), san-gan, and diffit are all driven by the same loop.
Each source declares its own `n_list` since per-class budgets differ
(orig=360 images/class, san=100k → sample down to 1k/10k, diffit-256=10k, diffit-512=1k).

In [ ]:
types_dict = {
    "Ultra_Co11": "средние зерна",
    "Ultra_Co25": "мелкие зерна",
    "Ultra_Co8": "средне-мелкие зерна",
    "Ultra_Co6_2": "крупные зерна",
    "Ultra_Co15": "средне-мелкие зерна",
}

min_segment_len = 5.0
output_dir = Path("./grid_results")
output_dir.mkdir(exist_ok=True)

# (path, n_list) — n_list is per-source so each generator's available
# sample budget is honoured. Comment a row to skip its (re)generation.
SOURCES = [
    # --- Original (ethalon) — N=360 images/class
    ('/home/david/mnt/ssd_2_sata/python/phd/datasets/san/o_bc_left_4x_1536_1024x1024_256x256_rgb_N360',   [360]),
    ('/home/david/mnt/ssd_2_sata/python/phd/datasets/san/o_bc_left_4x_1536_1024x1024_512x512_rgb_N360',   [360]),
    ('/home/david/mnt/ssd_2_sata/python/phd/datasets/san/o_bc_left_4x_1536_1024x1024_1024x1024_rgb_N360', [360]),

    # --- SAN-GAN — sample down from 100k
    ('./data/generated_images_h5/gen_san_256x256_N100_000.h5', [1_000, 10_000]),
    ('./data/generated_images_h5/gen_san_512x512_N100_000.h5', [1_000, 10_000]),
    # --- Diffit
    ('./data/generated_images_h5/00017-diffit-256-gpus2-batch192_N10000.h5', [1_000, 10_000]),
    ('./data/generated_images_h5/00018-diffit-512-gpus4-batch256_N1000.h5',  [1_000]),
]

for source_path, n_list in tqdm(SOURCES, desc='sources'):
    source_name = Path(source_path).stem
    save_path = output_dir / f"{source_name}_msl{int(min_segment_len)}"
    for max_n in tqdm(n_list, desc=source_name, leave=False):
        dataset = data.PobeditDataset(path=source_path, max_images_num_per_class=max_n)
        out = dataset.generate_angles(
            save_path=str(save_path),
            types_dict=types_dict,
            step=[0.1, 0.5, 1, 2, 3, 4, 5],
            workers=20,
            angles_tol=3,
            min_segment_len=min_segment_len,
            keep_contours=False,
            chunksize=64,
        )
        print(f"[{source_name}, N={max_n}] -> {out}")

# Grid of plots

In [ ]:
# Grid plot: orig vs generated (san-gan or diffit) at each (resolution × grain class).
# Pick which generated source to overlay on top of the orig ethalon.

from combra.metrics.compare import compare_pairs

GRID_DIR = Path('./grid_results')

# Switch between san-gan and diffit overlays.
gen_mode = 'diffit'  # 'gan' or 'diffit'

# Generated parquets store class_0/1/2 — the Ultra_Co* → class_N mapping
# differs between san and diffit because the two trainings indexed classes
# in different orders (diffit ordering verified against 2_comparison.ipynb;
# san ordering inherited from the original notebook).
GEN_NAME_FOR_PER_MODE = {
    'gan': {
        'Ultra_Co25': 'class_0',
        'Ultra_Co11': 'class_1',
        'Ultra_Co6_2': 'class_2',
    },
    'diffit': {
        'Ultra_Co11': 'class_0',
        'Ultra_Co25': 'class_1',
        'Ultra_Co6_2': 'class_2',
    },
}
GEN_NAME_FOR = GEN_NAME_FOR_PER_MODE[gen_mode]

# Per-resolution orig (ethalon) folder — each resolution has its own source.
ORIG_FOLDER_FOR = {
    256: 'o_bc_left_4x_1536_1024x1024_256x256_rgb_N360_msl5',
    512: 'o_bc_left_4x_1536_1024x1024_512x512_rgb_N360_msl5',
    1024: 'o_bc_left_4x_1536_1024x1024_1024x1024_rgb_N360_msl5',
}

# San-GAN folders — only 256/512 available.
GAN_FOLDER_FOR = {
    256: 'gen_san_256x256_N100_000_msl5',
    512: 'gen_san_512x512_N100_000_msl5',
}

# Diffit folders — only 256/512 available.
DIFFIT_FOLDER_FOR = {
    256: '00017-diffit-256-gpus2-batch192_N10000_msl5',
    512: '00018-diffit-512-gpus4-batch256_N1000_msl5',
}

# Available N values per resolution per generator (limits the outer loop).
GEN_AVAILABLE_N = {
    'gan':    {256: [1_000, 10_000], 512: [1_000, 10_000]},
    'diffit': {256: [1_000, 10_000], 512: [1_000]},
}

GEN_FOLDER_FOR = {'gan': GAN_FOLDER_FOR, 'diffit': DIFFIT_FOLDER_FOR}[gen_mode]

STYLES = {
    'orig':   dict(color='blue',   marker='circle'),
    'gan':    dict(color='orange', marker='square'),
    'diffit': dict(color='green',  marker='diamond'),
}

resolutions = [256, 512, 1024]
grain_classes = [
    ('Ultra_Co25', 'мелкие зерна'),
    ('Ultra_Co11', 'средние зерна'),
    ('Ultra_Co6_2', 'крупные зерна'),
]

step = 2
save = True
ylim = [0, 0.03]

for max_n in [1_000, 10_000]:
    grid = []
    metric_pairs = []  # one entry per resolution that has a gen overlay
    for res in resolutions:
        orig_pq = str(GRID_DIR / ORIG_FOLDER_FOR[res] / 'angles_n360.parquet')

        # Resolve gen parquet for this (res, max_n) — may be unavailable.
        gen_pq = None
        if res in GEN_FOLDER_FOR and max_n in GEN_AVAILABLE_N[gen_mode].get(res, []):
            candidate = GRID_DIR / GEN_FOLDER_FOR[res] / f'angles_n{max_n}.parquet'
            if candidate.exists():
                gen_pq = str(candidate)

        # Build per-class grid traces.
        row = []
        for class_key, _ in grain_classes:
            cell = [{
                'parquet': orig_pq,
                'class_name': f'class_{class_key}',
                'label': 'orig', **STYLES['orig'],
            }]
            if gen_pq is not None:
                cell.append({
                    'parquet': gen_pq,
                    'class_name': GEN_NAME_FOR[class_key],
                    'label': gen_mode, **STYLES[gen_mode],
                })
            row.append(cell)
        grid.append(row)

        # Queue this (res) for the metrics print, if a gen overlay exists.
        if gen_pq is not None:
            class_map = {f'class_{ck}': GEN_NAME_FOR[ck] for ck, _ in grain_classes}
            metric_pairs.append((str(res), orig_pq, gen_pq, class_map))

    if metric_pairs:
        print(f"\n=== {gen_mode}  N/class={max_n}  step={step} ===")
        compare_pairs(metric_pairs, step=step, coef=1000, label_header='res')

    angles.angles_plot_grid(
        grid=grid,
        row_titles=[f'{r}×{r}' for r in resolutions],
        col_titles=[label for _, label in grain_classes],
        step=step,
        ylim=ylim,
        title=f'Распределения углов ({gen_mode}, step={step}, N изобр. на класс={max_n})',
        save=f'angles_grid_{gen_mode}_n{max_n}_step{step}.png' if save else None,
    )